# 03 — SQLAlchemy ORM 2.0

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- définir un modèle ORM avec `DeclarativeBase` et `Mapped`
- manipuler des `Session` et le pattern unit of work
- définir des `relationship` et naviguer entre objets
- comprendre la différence lazy/eager loading

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- SQL et sqlite3 (notebook 01)
- SQLAlchemy Core (notebook 02)
- classes, dataclasses, type hints

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- Alembic (notebook 04)

## Plan

1. L'ORM en une phrase
2. Modèle déclaratif 2.0
3. Créer les tables
4. Sessions et unit of work
5. `relationship` et navigation
6. Requêtes ORM avec `select`
7. Lazy vs eager loading
8. Synthèse
9. Exercices

---

## 1. L'ORM en une phrase

L'ORM **mappe** vos classes Python sur des tables SQL. Chaque instance est une ligne, chaque attribut `Mapped` est une colonne. Les relations entre objets correspondent aux clés étrangères.

---

## 2. Modèle déclaratif 2.0

La syntaxe SQLAlchemy 2.0 utilise `DeclarativeBase` et `Mapped[T]`.

In [ ]:
from sqlalchemy import create_engine, String, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship


class Base(DeclarativeBase):
    pass


class Salle(Base):
    __tablename__ = 'salle'

    id: Mapped[int] = mapped_column(primary_key=True)
    nom: Mapped[str] = mapped_column(String(50), unique=True)
    capacite: Mapped[int]

    reservations: Mapped[list['Reservation']] = relationship(back_populates='salle')

    def __repr__(self) -> str:
        return f'Salle(id={self.id}, nom={self.nom!r})'


In [ ]:
class Reservation(Base):
    __tablename__ = 'reservation'

    id: Mapped[int] = mapped_column(primary_key=True)
    salle_id: Mapped[int] = mapped_column(ForeignKey('salle.id'))
    creneau: Mapped[str] = mapped_column(String(50))
    organisateur: Mapped[str] = mapped_column(String(100))

    salle: Mapped['Salle'] = relationship(back_populates='reservations')

    def __repr__(self) -> str:
        return f'Reservation(id={self.id}, creneau={self.creneau!r})'


---

## 3. Créer les tables

In [ ]:
engine = create_engine('sqlite:///:memory:')
Base.metadata.create_all(engine)


---

## 4. Sessions et unit of work

In [ ]:
from sqlalchemy.orm import Session

with Session(engine) as session:
    mars = Salle(nom='Mars', capacite=12)
    venus = Salle(nom='Venus', capacite=6)
    session.add_all([mars, venus])
    session.commit()
    print(mars.id, venus.id)


`session.add()` enregistre l'objet dans l'unité de travail. `session.commit()` envoie toutes les modifications en une seule transaction. `session.rollback()` annule.

---

## 5. `relationship` et navigation

In [ ]:
with Session(engine) as session:
    mars = session.execute(
        __import__('sqlalchemy').select(Salle).where(Salle.nom == 'Mars')
    ).scalar_one()
    r = Reservation(salle=mars, creneau='lundi 9h', organisateur='Alice')
    session.add(r)
    session.commit()
    print(mars.reservations)


---

## 6. Requêtes ORM avec `select`

In [ ]:
from sqlalchemy import select

with Session(engine) as session:
    stmt = select(Salle).where(Salle.capacite >= 10).order_by(Salle.nom)
    for s in session.scalars(stmt):
        print(s)


---

## 7. Lazy vs eager loading

Par défaut, les `relationship` sont **lazy** : l'accès déclenche un SELECT. Pour des raisons de performance, on peut activer le **eager loading**.

In [ ]:
from sqlalchemy.orm import selectinload

with Session(engine) as session:
    stmt = select(Salle).options(selectinload(Salle.reservations))
    for s in session.scalars(stmt):
        print(s.nom, s.reservations)


---

## Synthèse

| Concept ORM | Rôle |
|---|---|
| `DeclarativeBase` | Classe de base pour les modèles |
| `Mapped[T]` | Annotation de colonne typée |
| `mapped_column(...)` | Configuration de colonne |
| `relationship(...)` | Navigation entre tables |
| `Session` | Unité de travail, transactions |
| `session.add()` / `commit()` | Cycle de vie des objets |


### Règles à retenir

1. **`Mapped[T]` + `mapped_column()`** : c'est la syntaxe 2.0, pas `Column()` dans le corps de la classe.
2. **`Session(engine)` avec `with`** pour fermer proprement.
3. **Eager loading (`selectinload`)** pour éviter le N+1 query problem.
4. **L'ORM n'est pas magique** : comprenez le SQL qu'il génère (`echo=True`).

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Modèle ORM *(facile)*

Définir un modèle `Utilisateur` (id, nom, email) en ORM 2.0. Créer la table, insérer 2 utilisateurs, les lire.

In [1]:
# Votre code ici
from sqlalchemy import create_engine, String
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session


class Base(DeclarativeBase):
    pass


class Utilisateur(Base):
    __tablename__ = "utilisateurs"

    id: Mapped[int] = mapped_column(primary_key=True)
    nom: Mapped[str] = mapped_column(String)
    email: Mapped[str] = mapped_column(String)


engine = create_engine("sqlite:///test.db")
Base.metadata.create_all(engine)

with Session(engine) as session:
    session.add_all([
        Utilisateur(nom="Aled", email="Aled@mail.com"),
        Utilisateur(nom="Oskour", email="oskour@mail.com")
    ])
    session.commit()

    for user in session.query(Utilisateur).all():
        print(user.id, user.nom, user.email)

1 Aled Aled@mail.com
2 Oskour oskour@mail.com


In [2]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_SQLAlchemy_ORM", exercice=1)


📝 Exercice 1 marqué comme tenté 🟢


<details>
<summary>📖 Voir la correction</summary>

```python
from sqlalchemy import create_engine, String, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

class Base(DeclarativeBase): pass

class Utilisateur(Base):
    __tablename__ = 'utilisateur'
    id: Mapped[int] = mapped_column(primary_key=True)
    nom: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200))

engine = create_engine('sqlite:///:memory:')
Base.metadata.create_all(engine)

with Session(engine) as s:
    s.add_all([Utilisateur(nom='Alice', email='a@e'), Utilisateur(nom='Bob', email='b@e')])
    s.commit()
    for u in s.scalars(select(Utilisateur)):
        print(u.nom, u.email)
```

</details>

### Exercice 2 — Relationships *(moyen)*

Ajouter `Commande(id, utilisateur_id, produit: str)` avec un relationship vers `Utilisateur`. Naviguer de l'utilisateur vers ses commandes.

In [10]:
from sqlalchemy import create_engine, String, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session


class Base(DeclarativeBase):
    pass


class Utilisateur(Base):
    __tablename__ = "utilisateurs"

    id: Mapped[int] = mapped_column(primary_key=True)
    nom: Mapped[str] = mapped_column(String)

    commandes = relationship("Commande")


class Commande(Base):
    __tablename__ = "commandes"

    id: Mapped[int] = mapped_column(primary_key=True)
    produit: Mapped[str] = mapped_column(String)
    utilisateur_id: Mapped[int] = mapped_column(ForeignKey("utilisateurs.id"))


engine = create_engine("sqlite:///test2.db")
Base.metadata.create_all(engine)

with Session(engine) as session:
    user = Utilisateur(nom="Toto")
    user.commandes = [
        Commande(produit="Clavier"),
        Commande(produit="Souris")
    ]

    session.add(user)
    session.commit()

    utilisateur = session.query(Utilisateur).first()

    print(utilisateur.nom)
    for commande in utilisateur.commandes:
        print(commande.produit)


Toto
Clavier
Souris


In [9]:
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

In [5]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_SQLAlchemy_ORM", exercice=2)


📝 Exercice 2 marqué comme tenté 🟢


<details>
<summary>📖 Voir la correction</summary>

```python
from sqlalchemy import ForeignKey

class Commande(Base):
    __tablename__ = 'commande'
    id: Mapped[int] = mapped_column(primary_key=True)
    utilisateur_id: Mapped[int] = mapped_column(ForeignKey('utilisateur.id'))
    produit: Mapped[str] = mapped_column(String(100))
    utilisateur: Mapped['Utilisateur'] = relationship()

Base.metadata.create_all(engine)

with Session(engine) as s:
    alice = s.scalars(select(Utilisateur).where(Utilisateur.nom == 'Alice')).first()
    s.add(Commande(utilisateur=alice, produit='pain'))
    s.commit()
```

</details>

### Exercice 3 — CRUD complet fil rouge *(difficile)*

Écrire `creer_reservation(session, salle_nom, creneau, org)` et `lister_reservations(session, salle_nom)` en ORM. Gérer le cas où la salle n'existe pas.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_SQLAlchemy_ORM", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
def creer_reservation(session: Session, salle_nom: str, creneau: str, org: str) -> Reservation:
    salle = session.scalars(select(Salle).where(Salle.nom == salle_nom)).first()
    if not salle:
        raise ValueError(f'salle {salle_nom!r} inconnue')
    r = Reservation(salle=salle, creneau=creneau, organisateur=org)
    session.add(r)
    session.commit()
    return r

def lister_reservations(session: Session, salle_nom: str) -> list[Reservation]:
    return list(session.scalars(
        select(Reservation).join(Salle).where(Salle.nom == salle_nom)
    ))
```

</details>

---

## Ressources externes

### Documentation officielle
- [SQLAlchemy 2.0 ORM Tutorial](https://docs.sqlalchemy.org/en/20/orm/)
- [Mapped Column Configuration](https://docs.sqlalchemy.org/en/20/orm/mapped_attributes.html)